In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [2]:
# data = pd.read_stata(r"Z:\harmonized\ECU\ENEMDU\data_arm\ECU_1990m11_BID.dta") # para bases de stata
data = pd.read_stata(r"datos/ECU_2000m11_BID.dta") # para bases de stata

In [7]:
df, meta = pd.read_stata(r"datos/ECU_2000m11_BID.dta", iterator=True), None
meta = df.variable_labels()
print("\nVariable labels:")
for col, label in meta.items():
    print(f"{col}: {label}")


Variable labels:
region_BID_c: Regiones BID
region_c: 
pais_c: Nombre del PaÃ­s
anio_c: Anio de la encuesta
mes_c: Mes de la encuesta
zona_c: Zona del pais
factor_ch: Factor de expansion del hogar
idh_ch: ID del hogar
idp_ci: ID de la persona en el hogar
factor_ci: Factor de expansion del individuo
sexo_ci: Sexo del individuo
edad_ci: Edad del individuo en aÃ±os
relacion_ci: Relacion o parentesco con el jefe del hogar
civil_ci: Estado civil
jefe_ci: Jefe/a de hogar
nconyuges_ch: # de conyuges en el hogar
nhijos_ch: # de hijos en el hogar
notropari_ch: # de otros familiares en el hogar
notronopari_ch: # de no familiares en el hogar
nempdom_ch: # de empleados domesticos
clasehog_ch: Tipo de hogar
nmiembros_ch: # de miembros en el hogar
miembros_ci: =1: es miembro del hogar
nmayor21_ch: # de familiares mayores a 21 anios en el hogar
nmenor21_ch: # de familiares menores a 21 anios en el hogar
nmayor65_ch: # de familiares mayores a 65 anios en el hogar
nmenor6_ch: # de familiares menores a

## Revisar los datos

- rgnal - regional
- area - area
- rn - región natural
- cuidad - ciudad
- zona - zona
- sector - sector
- vivienda - vivienda
- hogar - hogar
- persona - persona
- numpers - número de personas
- edad - edad
- ingpat - Ingresos como patrono o cuenta propia
- ingasa - ingreso líquido por salario
- ingasa1 - cuál fue su ingreso total por sueldo
- asa - recibio por trabajo especies, alimentos etc.
- ingasa2 - monto recibido por trabajo especies
- ingsec - ocup. secundaria cual fué su ingreso por sueldo
- inginv - monto de ingresos derivados de bienes de capital
- ingjub - Ingresos por jubilación o pensión
- ingotr - por otros ingresos
- ingbon - ingreso por bono solidario
- fexp - factor de expansión
- ingrl - ingresos

En esta encuesta desaparecen 'ingasg' e 'ingepv' que eran el ingreso laboral monetario, desaparece también 'ingdom', y aparecen las variables 'ingasa', 'ingasa1', 'asa', 'ingasa2' e 'ingsec' qeu son ingresos del trabajo asalariado de la actividad primaria y secundaria monetario y no monetario, se manteine ingrl como ingreso total, para mantener la misma línea que con los años anteriores vamos solo a usar el ingreso del trabajo monetario de la actividad principal

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, se puede usar estas variables y el ingreso laboral asumiendo que cuando estaba trabajando tenía ese ingreso para intentar aproximar el salario mensual y de ahí el salario trimestral, esto solo funciona así ya que no tenemos una variable que explicite el mes, en encuestas que tengan el mes o trimestre explícito esto no sería igual.

Hay un problema con las variables 'ingasa' e 'ingasa1', los promedios son demasiado diferentes, puede ser que 'ingasa1' tenga valores en sucres aún cuando la entrevista ya se hizo en dólares, así que usaremos 'ingasa', dejamos por fuera el ingreso de salarios de la actividad secundaria por motivos de mantener comparable con los años anteriores 

In [12]:
data[['ingasa', 'ingasa1', 'ingsec']].mean()

ingasa         20.540662
ingasa1    552770.885090
ingsec          1.006103
dtype: float64

In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62469 entries, 0 to 62468
Columns: 268 entries, region_BID_c to cpi
dtypes: category(88), float32(25), float64(106), int16(4), int32(1), int8(32), object(12)
memory usage: 70.1+ MB


Filtramos solo las columnas de interés para alivar el peso en la memoria

In [14]:
data.columns

Index(['region_BID_c', 'region_c', 'pais_c', 'anio_c', 'mes_c', 'zona_c',
       'factor_ch', 'idh_ch', 'idp_ci', 'factor_ci',
       ...
       'aguamejorada_ch', 'aguamide_ch', 'bano_ch', 'banoex_ch',
       'banomejorado_ch', 'sinbano_ch', 'aguatrat_ch', 'des1_ch', 'des2_ch',
       'cpi'],
      dtype='object', length=268)

In [16]:
data = data[['rgnal', 'area', 'rn', 'ciudad', 'zona', 'sector', 'vivienda',
             'hogar', 'persona', 'numpers', 'edad', 'ingpat', 'ingasa',
             'ingasa1', 'asa', 'ingasa2', 'ingsec', 'inginv', 'ingjub',
             'ingotr', 'ingbon', 'fexp', 'ingrl', 'ene', 'feb', 'mar', 
             'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic']]

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado en el sector público, asalariado en el sector privado o ingresos por empleo doméstico

In [17]:
data['ingr'] = data['ingasa']

/tmp/ipykernel_133083/1744984237.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr'] = data['ingasa']


Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados en un mes, ahora las variables categoricas por mes tienen diferentes leyendas
- desocupado y sin buscar trabajo
- trabajando
- buscando trabajo
- 0.0

In [21]:
data['ingr_ene'] = data.apply(lambda x: x['ingr'] if x['ene'] == 'trabajando' else None, axis=1)
data['ingr_feb'] = data.apply(lambda x: x['ingr'] if x['feb'] == 'trabajando' else None, axis=1)
data['ingr_mar'] = data.apply(lambda x: x['ingr'] if x['mar'] == 'trabajando' else None, axis=1)
data['ingr_abr'] = data.apply(lambda x: x['ingr'] if x['abr'] == 'trabajando' else None, axis=1)
data['ingr_may'] = data.apply(lambda x: x['ingr'] if x['may'] == 'trabajando' else None, axis=1)
data['ingr_jun'] = data.apply(lambda x: x['ingr'] if x['jun'] == 'trabajando' else None, axis=1)
data['ingr_jul'] = data.apply(lambda x: x['ingr'] if x['jul'] == 'trabajando' else None, axis=1)
data['ingr_ago'] = data.apply(lambda x: x['ingr'] if x['ago'] == 'trabajando' else None, axis=1)
data['ingr_sep'] = data.apply(lambda x: x['ingr'] if x['sep'] == 'trabajando' else None, axis=1)
data['ingr_oct'] = data.apply(lambda x: x['ingr'] if x['oct'] == 'trabajando' else None, axis=1)
data['ingr_nov'] = data.apply(lambda x: x['ingr'] if x['nov'] == 'trabajando' else None, axis=1)
data['ingr_dic'] = data.apply(lambda x: x['ingr'] if x['dic'] == 'trabajando' else None, axis=1)

/tmp/ipykernel_133083/2782776572.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_ene'] = data.apply(lambda x: x['ingr'] if x['ene'] == 'trabajando' else None, axis=1)
/tmp/ipykernel_133083/2782776572.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_feb'] = data.apply(lambda x: x['ingr'] if x['feb'] == 'trabajando' else None, axis=1)
/tmp/ipykernel_133083/2782776572.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[ro

Ingreso mensual promedio en el trimeste

In [22]:
data['ingr_t1'] = (data['ingr_ene'] + data['ingr_feb'] + data['ingr_mar'])/3
data['ingr_t2'] = (data['ingr_abr'] + data['ingr_may'] + data['ingr_jun'])/3
data['ingr_t3'] = (data['ingr_jul'] + data['ingr_ago'] + data['ingr_sep'])/3
data['ingr_t4'] = (data['ingr_oct'] + data['ingr_nov'] + data['ingr_dic'])/3

/tmp/ipykernel_133083/2345085531.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_t1'] = (data['ingr_ene'] + data['ingr_feb'] + data['ingr_mar'])/3
/tmp/ipykernel_133083/2345085531.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_t2'] = (data['ingr_abr'] + data['ingr_may'] + data['ingr_jun'])/3
/tmp/ipykernel_133083/2345085531.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead



## Corregimos los valores de ser necesario

In [23]:
# Llenar valores perdidos con un dato (0)
data['ingr_t1'] = data['ingr_t1'].fillna(0)
data['ingr_t2'] = data['ingr_t2'].fillna(0)
data['ingr_t3'] = data['ingr_t3'].fillna(0)
data['ingr_t4'] = data['ingr_t4'].fillna(0)

/tmp/ipykernel_133083/1998444320.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_t1'] = data['ingr_t1'].fillna(0)
/tmp/ipykernel_133083/1998444320.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_t2'] = data['ingr_t2'].fillna(0)
/tmp/ipykernel_133083/1998444320.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pa

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [24]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2000]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc y tipo de cambio

In [25]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_base.iterrows()
     }

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [28]:
# Corregimos los códigos para usarlos cómo texto
data['ciudad'] = data['ciudad'].apply(str)
data['ciudad'] = data['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data['ciudad_2'] = data['ciudad'].apply(lambda x: x[:2])

/tmp/ipykernel_133083/2908473067.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ciudad'] = data['ciudad'].apply(str)
/tmp/ipykernel_133083/2908473067.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ciudad'] = data['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
/tmp/ipykernel_133083/2908473067.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documenta

Diccionario ciudades disponibles

In [29]:
parroquia_dict = {
    '01': 'Cuenca',
    '09': 'Guayaquil',
    '17': 'Quito'
}

data['ciudad_asignada'] = data['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))

/tmp/ipykernel_133083/218289303.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ciudad_asignada'] = data['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))


Elegimos el trimestre según las variables del INEC donde sea igual a 'trabajando'

In [32]:
def assgna_trimestre(fila):
    if 'trabajando' in [fila['ene'], fila['feb'], fila['mar']]:
        return 1
    elif 'trabajando' in [fila['abr'], fila['may'], fila['jun']]:
        return 2
    elif 'trabajando' in [fila['jul'], fila['ago'], fila['sep']]:
        return 3
    elif 'trabajando' in [fila['oct'], fila['nov'], fila['dic']]:
        return 4
    else:
        return None
    
data['trimestre'] = data.apply(assgna_trimestre, axis=1)

/tmp/ipykernel_133083/2631173444.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['trimestre'] = data.apply(assgna_trimestre, axis=1)


### Asignamos el ipc correspondiente según trimestre y ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = ingr_{dólares}^{i}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Desde 2000 en adelante ya no es necesario utilizar el tipo de cambio debido al cambio de moneda

In [33]:
# Función que asigna valores correspondientes
def asigna_ipc(fila):
    return ipc_dict.get(fila['trimestre'], {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila):
    return ipc_base_dict.get(fila['trimestre'], {}).get(fila['ciudad_asignada'], None)

In [34]:
data['ipc'] = data.apply(asigna_ipc, axis=1)
data['ipc_base'] = data.apply(asigna_ipc_base, axis=1)

/tmp/ipykernel_133083/1240146704.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ipc'] = data.apply(asigna_ipc, axis=1)
/tmp/ipykernel_133083/1240146704.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ipc_base'] = data.apply(asigna_ipc_base, axis=1)


In [35]:
# Calculamos el deflactor
data['def'] = (data['ipc_base'] / data['ipc'])

/tmp/ipykernel_133083/587192262.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['def'] = (data['ipc_base'] / data['ipc'])


In [36]:
data['ingr_t1_r'] = data['ingr_t1'] * data['def']
data['ingr_t2_r'] = data['ingr_t2'] * data['def']
data['ingr_t3_r'] = data['ingr_t3'] * data['def']
data['ingr_t4_r'] = data['ingr_t4'] * data['def']

/tmp/ipykernel_133083/3212223615.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_t1_r'] = data['ingr_t1'] * data['def']
/tmp/ipykernel_133083/3212223615.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_t2_r'] = data['ingr_t2'] * data['def']
/tmp/ipykernel_133083/3212223615.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.p

In [38]:
data[['ingr_t1', 'ingr_t2', 'ingr_t3', 'ingr_t4', 'ipc', 'ipc_base', 'def', 'ingr_t1_r', 'ingr_t2_r', 'ingr_t3_r', 'ingr_t4_r']]

,ingr_t1,ingr_t2,ingr_t3,ingr_t4,ipc,ipc_base,def,ingr_t1_r,ingr_t2_r,ingr_t3_r,ingr_t4_r
0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.0,0.0,0.0,0.0,36.055094,97.862086,2.714237,0.000000,0.000000,0.000000,0.000000
3,0.0,0.0,0.0,0.0,36.055094,97.862086,2.714237,0.000000,0.000000,0.000000,0.000000
4,0.0,0.0,0.0,0.0,36.055094,97.862086,2.714237,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
62464,0.0,0.0,0.0,0.0,38.871625,98.083034,2.523255,0.000000,0.000000,0.000000,0.000000
62465,24.0,24.0,24.0,24.0,38.871625,98.083034,2.523255,60.558127,60.558127,60.558127,60.558127
62466,0.0,0.0,0.0,0.0,38.871625,98.083034,2.523255,0.000000,0.000000,0.000000,0.000000
62467,24.0,24.0,24.0,24.0,38.871625,98.083034,2.523255,60.558127,60.558127,60.558127,60.558127


## Calculo ingreso de los hogares

In [39]:
columnas_idef = ['rgnal', 'area', 'rn', 'ciudad', 'zona', 'sector', 'vivienda',
             'hogar']

data['idef_hogar'] = data[columnas_idef].astype(str).agg(''.join, axis=1)
len(data['idef_hogar'].unique())

/tmp/ipykernel_133083/1930206229.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['idef_hogar'] = data[columnas_idef].astype(str).agg(''.join, axis=1)


13963

In [40]:
data[['rgnal', 'area', 'rn', 'ciudad', 'zona', 'sector', 'vivienda',
             'hogar', 'idef_hogar', 'persona', 'numpers']]

,rgnal,area,rn,ciudad,zona,sector,vivienda,hogar,idef_hogar,persona,numpers
0,4,1,1,010150,001,004,01,1,411010150001004011,4,4
1,4,1,1,010150,001,004,01,1,411010150001004011,3,4
2,4,1,1,010150,001,004,01,1,411010150001004011,1,4
3,4,1,1,010150,001,004,01,1,411010150001004011,2,4
4,4,1,1,010150,001,004,09,1,411010150001004091,2,6
...,...,...,...,...,...,...,...,...,...,...,...
62464,3,2,1,050160,999,015,07,1,321050160999015071,1,5
62465,3,2,1,050160,999,015,07,1,321050160999015071,4,5
62466,3,2,1,050160,999,015,07,1,321050160999015071,5,5
62467,3,2,1,050160,999,015,07,1,321050160999015071,3,5


In [41]:
data['ingr_t1_h'] = data.groupby('idef_hogar')['ingr_t1_r'].transform('sum')
data['ingr_t2_h'] = data.groupby('idef_hogar')['ingr_t2_r'].transform('sum')
data['ingr_t3_h'] = data.groupby('idef_hogar')['ingr_t3_r'].transform('sum')
data['ingr_t4_h'] = data.groupby('idef_hogar')['ingr_t4_r'].transform('sum')

/tmp/ipykernel_133083/3037391686.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_t1_h'] = data.groupby('idef_hogar')['ingr_t1_r'].transform('sum')
/tmp/ipykernel_133083/3037391686.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_t2_h'] = data.groupby('idef_hogar')['ingr_t2_r'].transform('sum')
/tmp/ipykernel_133083/3037391686.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead



In [42]:
data[['ingr_t1_h', 'ingr_t2_h', 'ingr_t3_h', 'ingr_t4_h']]

,ingr_t1_h,ingr_t2_h,ingr_t3_h,ingr_t4_h
0,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...
62464,121.116254,121.116254,121.116254,121.116254
62465,121.116254,121.116254,121.116254,121.116254
62466,121.116254,121.116254,121.116254,121.116254
62467,121.116254,121.116254,121.116254,121.116254


In [43]:
print("Ingreso medio de un hogar t4: ", data['ingr_t4_h'].mean())
print("Mediana del ingreso de un hogar t4: ", data['ingr_t4_h'].median())

Ingreso medio de un hogar t4:  207.37938228000198
Mediana del ingreso de un hogar t4:  100.93021194959621


## Sacamos edades negativas y mayores a 100 años

In [44]:
len(data)

62469

En este caso en la variable edad tenemos números y el texto 'menos de un año' así que primero transformamos todas las filas que digan 'menos de un año' a 0

In [45]:
data['edad'] = data['edad'].apply(lambda x: x if type(x) == int else 0)

/tmp/ipykernel_133083/3957095988.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['edad'] = data['edad'].apply(lambda x: x if type(x) == int else 0)


In [46]:
data = data.loc[(data['edad'] >= 0) & (data['edad'] < 100)]
len(data)

62469

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [47]:
k = 0.4
s = 0.9

In [48]:
# Si es necesario calcular el número de niños
data['es_nino'] = data['edad'] < 10

data['ninos'] = data.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data['es_adulto'] = data['edad'] > 10

data['adultos'] = data.groupby('idef_hogar')['es_adulto'].transform('sum')

In [49]:
data['escala'] = (data['adultos'] + k * data['ninos']) ** s

In [50]:
data['ingr_t_t1'] = data['ingr_t1_h'] / data['escala']
data['ingr_t_t2'] = data['ingr_t2_h'] / data['escala']
data['ingr_t_t3'] = data['ingr_t3_h'] / data['escala']
data['ingr_t_t4'] = data['ingr_t4_h'] / data['escala']

In [51]:
data[['ingr_t_t1', 'ingr_t_t2', 'ingr_t_t3', 'ingr_t_t4']]

,ingr_t_t1,ingr_t_t2,ingr_t_t3,ingr_t_t4
0,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...
62464,28.453089,28.453089,28.453089,28.453089
62465,28.453089,28.453089,28.453089,28.453089
62466,28.453089,28.453089,28.453089,28.453089
62467,28.453089,28.453089,28.453089,28.453089


In [52]:
print("Ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].mean())
print("Mediana del ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].median())

Ingreso individual descontando cargas familiares t4:  55.81150573045728
Mediana del ingreso individual descontando cargas familiares t4:  24.59824210140199


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [53]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))

In [55]:
# Asigna el umbral por trimestre si hay umbral por región modificar
data['umbral'] = data['trimestre'].map(umbral_dict)

data['umbral'] = data.groupby('idef_hogar')['umbral'].transform('mean')

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

In [56]:
datos_final = pd.DataFrame(index=['t1', 't2', 't3', 't4'], columns=['fgt0', 'fgt1', 'fgt2', 'a25', 'a50', 'a75', 'ingreso_promedio'])

In [57]:
data['persona_fexp'] = 1 * data['fexp']

In [58]:
for t in [1, 2, 3, 4]:
    condicion = data['trimestre'] == t
    col_ingr = f'ingr_t_t{t}'
    col_pobres = f'pobres_t{t}'

    # una columna que identifica a quienes están por debajo de la línea de pobreza por trimestre
    data.loc[condicion, col_pobres] = (
        (data.loc[condicion, col_ingr] - data.loc[condicion, 'umbral']) < 0
    ).astype(int)

In [59]:
print("pobreza t1: ", (data['pobres_t1'] * data['fexp']).sum()/data.loc[data['trimestre'] == 1]['persona_fexp'].sum())
print("pobreza t2: ", (data['pobres_t2'] * data['fexp']).sum()/data.loc[data['trimestre'] == 2]['persona_fexp'].sum())
print("pobreza t3: ", (data['pobres_t3'] * data['fexp']).sum()/data.loc[data['trimestre'] == 3]['persona_fexp'].sum())
print("pobreza t4: ", (data['pobres_t4'] * data['fexp']).sum()/data.loc[data['trimestre'] == 4]['persona_fexp'].sum())

pobreza t1:  0.8025751813475804
pobreza t2:  0.8881593329555302
pobreza t3:  0.8774169074673801
pobreza t4:  0.8874634670207704


In [60]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data['trimestre'] == t].copy()

    # Calculamos una columna de pobres
    df_temp['pobres'] = (df_temp[f'ingr_t_t{t}'] - umbral_dict[t]) < 0

    # Ratio de pobres sobre el total
    ratio = (umbral_dict[t] - df_temp[f'ingr_t_t{t}']) / umbral_dict[t]

    # Calculamos el índice para alpha 0, 1 y 2 solo donde 'pobres' == True.
    for i in range(3):
        col = f'fgt{i}'
        df_temp[col] = np.where(df_temp['pobres'], ratio**i, 0)

    # Cálculo del índice ponderado: se usa el factor de expansión como peso
    peso_total = df_temp['fexp'].sum()
    fgt0 = (df_temp['fgt0'] * df_temp['fexp']).sum() / peso_total
    fgt1 = (df_temp['fgt1'] * df_temp['fexp']).sum() / peso_total
    fgt2 = (df_temp['fgt2'] * df_temp['fexp']).sum() / peso_total
    
    # Guardamos los resultados
    datos_final.loc[f't{t}', 'fgt0'] = fgt0
    datos_final.loc[f't{t}', 'fgt1'] = fgt1
    datos_final.loc[f't{t}', 'fgt2'] = fgt2

In [61]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.802007,0.581413,0.497327,NaN,NaN,NaN,NaN
t2,0.888896,0.700136,0.614654,NaN,NaN,NaN,NaN
t3,0.878414,0.656679,0.564631,NaN,NaN,NaN,NaN
t4,0.890491,0.689239,0.596993,NaN,NaN,NaN,NaN


## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [62]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data['trimestre'] == t].copy()

    # Suma total de los factores de expansión para el trimestre
    peso_total = df_temp['fexp'].sum()
    
    # Ingreso promedio ponderado
    mu = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total

    # Calculamos el índice A para epsilon 0.25, 0.5 y 0.75 utilizando los pesos
    indices = {}
    for i in [0.25, 0.5, 0.75]:
        A_i = ((df_temp[f'ingr_t_t{t}']**(1-i) * df_temp['fexp']).sum() / peso_total)**(1/(1-i))
        indices[i] = A_i

    # Ratio de pobreza con el índice total (aplicando la fórmula)
    a25 = 1 - 1/mu * indices[0.25]
    a50 = 1 - 1/mu * indices[0.5]
    a75 = 1 - 1/mu * indices[0.75]

    # Guardamos los resultados
    datos_final.loc[f't{t}', 'a25'] = a25
    datos_final.loc[f't{t}', 'a50'] = a50
    datos_final.loc[f't{t}', 'a75'] = a75


In [63]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.802007,0.581413,0.497327,0.24467,0.498398,0.80888,NaN
t2,0.888896,0.700136,0.614654,0.279955,0.567428,0.878968,NaN
t3,0.878414,0.656679,0.564631,0.220889,0.478758,0.809439,NaN
t4,0.890491,0.689239,0.596993,0.240207,0.51763,0.849795,NaN


Guardamos el ingreso promedio

In [64]:
for t in [1, 2, 3, 4]:
    df_temp = data.loc[data['trimestre'] == t].copy()
    
    # Calcula la suma total de los factores de expansión
    peso_total = df_temp['fexp'].sum()
    
    # Calcula el ingreso promedio ponderado
    media_ponderada = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total
    
    datos_final.loc[f't{t}', 'ingreso_promedio'] = media_ponderada

In [65]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.802007,0.581413,0.497327,0.24467,0.498398,0.80888,74.800076
t2,0.888896,0.700136,0.614654,0.279955,0.567428,0.878968,47.186588
t3,0.878414,0.656679,0.564631,0.220889,0.478758,0.809439,47.70068
t4,0.890491,0.689239,0.596993,0.240207,0.51763,0.849795,45.325952


In [66]:
datos_final.to_csv('datos_final.csv')